In [82]:
import pandas as pd
import pubchempy as pcp

# --- 1) load and extract L1000 pubchem_cids, cast to int ---
broad_meta = pd.read_csv('broad.txt', sep='\t',
                         dtype={'pubchem_cid': str})

# coerce non-numeric strings → NaN, then drop them
broad_meta['pubchem_cid_numeric'] = pd.to_numeric(
    broad_meta['pubchem_cid'], errors='coerce'
)
L1000_cid_set = set(
    broad_meta['pubchem_cid_numeric']
      .dropna()
      .astype(int)   # now pure Python ints
)

# --- 2) load Tahoe and cast to int (you already did this) ---
Tahoe = pd.read_parquet('drug_metadata.parquet')
tahoe_cid_set = set(
    Tahoe['pubchem_cid']
         .dropna()
         .astype(int)
)

# --- 3) build SciPlex set, filtering out missing, as ints ---
sciplex_drugs = pd.read_csv('unique_perturbations.csv')
sciplex_cids = {}
for name in sciplex_drugs.iloc[:,0]:
    comps = pcp.get_compounds(name, 'name')
    sciplex_cids[name] = comps[0].cid if comps else None

sciplex_cid_set = set(
    val for val in sciplex_cids.values()
    if val is not None     # drop the Nones
)

# --- 4) intersections now work correctly ---
print("L1000 ∩ Tahoe  CIDs:", L1000_cid_set & tahoe_cid_set)
print("L1000 ∩ SciPlex CIDs:", L1000_cid_set & sciplex_cid_set)


L1000 ∩ Tahoe  CIDs: {2563, 57363, 5284373, 32798, 92727, 42611257, 23725625, 44093, 11626560, 60490, 31307, 451668, 23582824, 440936, 6253, 5743, 6256, 185462, 56959, 25183872, 4740, 6279, 9926791, 2187, 2194, 44462760, 77999, 6918837, 5388983, 11683005, 60606, 6442177, 2244, 204, 5344, 445154, 9444, 11707110, 2747117, 123631, 441074, 2812, 71420, 5284616, 3339, 12560, 5905, 60699, 68911, 11338033, 53235510, 119607, 3385, 104769, 24775005, 33630, 9915743, 285033, 5330286, 4463, 3440, 4659569, 370, 25262965, 387447, 11219835, 444795, 10621, 2435, 167811, 6019, 4495, 104850, 3034010, 441243, 5823908, 2478, 24826799, 392622, 5282230, 4030, 71616, 1986, 1989, 636362, 55245, 10184653, 238053, 16362, 6918638, 4594}
L1000 ∩ SciPlex CIDs: {448013, 2577, 11228183, 60953, 406563, 176167, 9914412, 3117, 3062316, 5328940, 24753719, 5281855, 11626560, 60490, 451668, 702558, 568416, 160355, 4713, 24978538, 1645, 167551, 24785538, 11712649, 29327, 644241, 2712, 9933475, 4261, 24748204, 10367662, 549

In [86]:
import scanpy as sc

In [97]:
adata = sc.read_h5ad('tahoe_a549_deg_adata.h5ad')
adata.obs['pubchem_cid'] = pd.to_numeric(adata.obs['pubchem_cid'], errors='coerce').astype('Int64')
mask1 = adata.obs['pubchem_cid'].isin(list(L1000_Tahoe_intersection))
mask_np = mask1.to_numpy(dtype=bool)
adata_sub = adata[mask_np].copy()

In [100]:
adata_sub.write_h5ad('tahoe_a549_deg_adata_drug_subset.h5ad')